# AIS Orientation Prediction Training

Train an image-based RPY delta predictor from `ws_aic/data/ais_rpy_randomization`.

The default target is the actual corrective delta from `actual.port_in_plug_reference`, in radians. Checkpoints are saved under `ws_aic/model/ais_orientation_prediction`.


In [1]:
from pathlib import Path
import sys


def find_project_dir():
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path('/home/whyz/aic_sejong/ws_aic/src/ais/ais_orientation_prediction'),
    ]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'model' / 'dataset.py').is_file():
            return candidate
        nested = candidate / 'ws_aic' / 'src' / 'ais' / 'ais_orientation_prediction'
        if (nested / 'model' / 'dataset.py').is_file():
            return nested.resolve()
    raise RuntimeError('Could not find src/ais/ais_orientation_prediction')


PROJECT_DIR = find_project_dir()
AIS_SRC = PROJECT_DIR.parent
WS_ROOT = PROJECT_DIR.parents[2]
if str(AIS_SRC) not in sys.path:
    sys.path.insert(0, str(AIS_SRC))

PROJECT_DIR, WS_ROOT, AIS_SRC

(PosixPath('/home/whyz/aic_sejong/ws_aic/src/ais/ais_orientation_prediction'),
 PosixPath('/home/whyz/aic_sejong/ws_aic'),
 PosixPath('/home/whyz/aic_sejong/ws_aic/src/ais'))

In [2]:
from collections import Counter
import math

import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from ais_orientation_prediction.model import (
    DEFAULT_DATASET_ROOT,
    DEFAULT_WEIGHT_ROOT,
    OrientationDeltaDataset,
    build_orientation_model,
    compute_target_stats,
    evaluate,
    fit,
    load_samples,
    sample_has_target,
    seed_everything,
    split_samples_by_episode,
    target_from_sample,
)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


/home/whyz/aic_sejong/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [3]:
DATASET_ROOT = DEFAULT_DATASET_ROOT
WEIGHT_ROOT = DEFAULT_WEIGHT_ROOT

VERSIONS = ['v3.1','v2.3']
CAMERAS = ('left', 'center', 'right')
TARGET_SOURCE = 'actual'
TARGET_MODE = 'correction'
TARGET_UNIT = 'rad'

IMAGE_SIZE = (256, 288)
BATCH_SIZE = 16
NUM_WORKERS = 8
EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-4
DIRECTION_LOSS_WEIGHT = 0.1
MAGNITUDE_LOSS_WEIGHT = 0.1
DIRECTION_EPS = 1e-3
EMA_DECAY = 0.999
BACKBONE = 'resnet50'
PRETRAINED = True
AGGREGATION = 'concat'
NUM_PORT_HEADS = 2
VAL_RATIO = 0.2

VERSION_TAG = '_'.join(VERSIONS)
SOURCE_TAG = f'{TARGET_SOURCE}_{TARGET_MODE}'
RUN_NAME = f"rpy_delta_{SOURCE_TAG}_{BACKBONE}_{'_'.join(CAMERAS)}_{AGGREGATION}_{VERSION_TAG}"

DATASET_ROOT, WEIGHT_ROOT / RUN_NAME


(PosixPath('/home/whyz/aic_sejong/ws_aic/data/ais_rpy_randomization'),
 PosixPath('/home/whyz/aic_sejong/ws_aic/model/ais_orientation_prediction/rpy_delta_actual_correction_resnet50_left_center_right_concat_v3.1_v2.3'))

In [4]:
samples = load_samples(DATASET_ROOT, versions=VERSIONS, splits=('train', 'val'))
if not samples:
    raise FileNotFoundError(f'No samples found under {DATASET_ROOT} for versions={VERSIONS}')

missing_files = []
missing_camera_counts = Counter()
for sample in samples:
    for camera in CAMERAS:
        image_relpath = sample.get('images', {}).get(camera)
        if image_relpath is None:
            missing_camera_counts[camera] += 1
            continue
        image_path = Path(sample['_dataset_root']) / image_relpath
        if not image_path.is_file():
            missing_files.append(image_path)
if missing_files:
    preview = '\n'.join(str(path) for path in missing_files[:5])
    raise FileNotFoundError(f'{len(missing_files)} image files are missing. First missing files:\n{preview}')

complete_samples = [
    sample
    for sample in samples
    if all(camera in sample.get('images', {}) for camera in CAMERAS)
    and sample_has_target(sample, target_source=TARGET_SOURCE, target_mode=TARGET_MODE)
]
if not complete_samples:
    raise RuntimeError(f'No complete {CAMERAS} samples found under {DATASET_ROOT} for versions={VERSIONS}')


def sample_version(sample):
    return sample.get('version') or sample.get('_version', '')


complete_by_version = Counter(sample_version(sample) for sample in complete_samples)
missing_versions = [version for version in VERSIONS if complete_by_version.get(version, 0) == 0]
if missing_versions:
    raise RuntimeError(f'No complete {CAMERAS} samples for versions: {missing_versions}')

train_samples, val_samples, _ = split_samples_by_episode(
    complete_samples,
    val_ratio=VAL_RATIO,
    test_ratio=0.0,
    seed=42,
)
if not train_samples or not val_samples:
    raise RuntimeError(f'Invalid split: train={len(train_samples)} val={len(val_samples)}')

val_samples_by_version = {
    version: [sample for sample in val_samples if sample_version(sample) == version]
    for version in VERSIONS
}
extra_val_sample_sets = {
    version.replace('.', '_'): rows
    for version, rows in val_samples_by_version.items()
    if rows
}

target_stats = compute_target_stats(
    train_samples,
    target_source=TARGET_SOURCE,
    target_mode=TARGET_MODE,
    target_unit=TARGET_UNIT,
)

def target_stack(rows):
    return torch.stack([
        target_from_sample(sample, target_source=TARGET_SOURCE, target_mode=TARGET_MODE, target_unit=TARGET_UNIT)
        for sample in rows
    ])


def angular_baselines(rows):
    targets = target_stack(rows)
    zero = torch.linalg.vector_norm(targets, dim=1).mean()
    mean = torch.linalg.vector_norm(targets - target_stats['mean'], dim=1).mean()
    return zero, mean

val_targets = target_stack(val_samples)
zero_baseline_angular, mean_baseline_angular = angular_baselines(val_samples)
version_baselines = {
    version: angular_baselines(rows)
    for version, rows in val_samples_by_version.items()
    if rows
}

print(f'dataset_root: {DATASET_ROOT}')
print(f'versions: {VERSIONS}')
print(f'target_source: {TARGET_SOURCE} target_mode: {TARGET_MODE}')
print(f'samples: total={len(samples)} complete_{len(CAMERAS)}view={len(complete_samples)} train={len(train_samples)} val={len(val_samples)}')
print(f'complete_by_version: {dict(complete_by_version)}')
print(f'train_by_version: {dict(Counter(sample_version(sample) for sample in train_samples))}')
print(f'val_by_version: {dict(Counter(sample_version(sample) for sample in val_samples))}')
print(f'missing_camera_counts: {dict(missing_camera_counts)}')
print(f'ports: {Counter(sample.get("port_name", "") for sample in samples)}')
print(f'target_mean_rad: {target_stats["mean"].tolist()}')
print(f'target_std_rad:  {target_stats["std"].tolist()}')
print(f'target_mean_deg: {(target_stats["mean"] * 180.0 / math.pi).tolist()}')
print(f'target_std_deg:  {(target_stats["std"] * 180.0 / math.pi).tolist()}')
print(f'zero_baseline_angular_deg: {(zero_baseline_angular * 180.0 / math.pi).item():.4f}')
print(f'mean_baseline_angular_deg: {(mean_baseline_angular * 180.0 / math.pi).item():.4f}')
for version, (zero, mean) in version_baselines.items():
    print(f'{version}_zero_baseline_angular_deg: {(zero * 180.0 / math.pi).item():.4f}')
    print(f'{version}_mean_baseline_angular_deg: {(mean * 180.0 / math.pi).item():.4f}')

dataset_root: /home/whyz/aic_sejong/ws_aic/data/ais_rpy_randomization
versions: ['v3.1', 'v2.3']
target_source: actual target_mode: correction
samples: total=2318 complete_3view=2175 train=1752 val=423
complete_by_version: {'v3.1': 1040, 'v2.3': 1135}
train_by_version: {'v3.1': 795, 'v2.3': 957}
val_by_version: {'v3.1': 245, 'v2.3': 178}
missing_camera_counts: {'right': 120, 'center': 71, 'left': 13}
ports: Counter({'sfp_port_0': 1234, 'sfp_port_1': 1084})
target_mean_rad: [-0.0002911576593760401, 0.0008384232060052454, 0.0029987029265612364]
target_std_rad:  [0.09025861322879791, 0.09257720410823822, 0.12801283597946167]
target_mean_deg: [-0.016682105138897896, 0.04803811013698578, 0.1718130260705948]
target_std_deg:  [5.1714372634887695, 5.3042826652526855, 7.334595203399658]
zero_baseline_angular_deg: 8.5406
mean_baseline_angular_deg: 8.5316
v3.1_zero_baseline_angular_deg: 13.6499
v3.1_mean_baseline_angular_deg: 13.6461
v2.3_zero_baseline_angular_deg: 1.5081
v2.3_mean_baseline_angul

In [5]:
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [6]:
train_dataset = OrientationDeltaDataset(
    DATASET_ROOT,
    samples=train_samples,
    cameras=CAMERAS,
    transform=train_transform,
    target_mode=TARGET_MODE,
    target_unit=TARGET_UNIT,
    target_source=TARGET_SOURCE,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)
val_dataset = OrientationDeltaDataset(
    DATASET_ROOT,
    samples=val_samples,
    cameras=CAMERAS,
    transform=eval_transform,
    target_mode=TARGET_MODE,
    target_unit=TARGET_UNIT,
    target_source=TARGET_SOURCE,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
)
extra_val_datasets = {
    name: OrientationDeltaDataset(
        DATASET_ROOT,
        samples=rows,
        cameras=CAMERAS,
        transform=eval_transform,
        target_mode=TARGET_MODE,
        target_unit=TARGET_UNIT,
        target_source=TARGET_SOURCE,
        target_mean=target_stats['mean'],
        target_std=target_stats['std'],
    )
    for name, rows in extra_val_sample_sets.items()
}
if len(train_dataset) == 0 or len(val_dataset) == 0:
    raise RuntimeError(f'No complete samples after camera filtering: train={len(train_dataset)} val={len(val_dataset)}')
print(f'complete {CAMERAS} samples: train={len(train_dataset)} val={len(val_dataset)}')
print('extra val:', {name: len(dataset) for name, dataset in extra_val_datasets.items()})

pin_memory = device.type == 'cuda'
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)
extra_val_loaders = {
    name: DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )
    for name, dataset in extra_val_datasets.items()
}

batch = next(iter(train_loader))
batch['image'].shape, batch['target'].shape, batch['raw_target'][:3], batch['port_id'][:8]

complete ('left', 'center', 'right') samples: train=1752 val=423
extra val: {'v3_1': 245, 'v2_3': 178}


(torch.Size([16, 3, 3, 256, 288]),
 torch.Size([16, 3]),
 tensor([[-0.0054,  0.0197, -0.0132],
         [-0.0166,  0.0019,  0.0228],
         [-0.0083,  0.1635,  0.1836]]),
 tensor([0, 0, 1, 0, 0, 0, 0, 0]))

In [7]:
model = build_orientation_model(
    backbone_name=BACKBONE,
    pretrained=PRETRAINED,
    output_dim=3,
    hidden_dim=256,
    dropout=0.1,
    aggregation=AGGREGATION,
    num_views=len(CAMERAS),
    num_port_heads=NUM_PORT_HEADS,
)
model.to(device)

optimizer = torch.optim.AdamW(
    (parameter for parameter in model.parameters() if parameter.requires_grad),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

26847181

In [8]:
config = {
    'dataset_root': str(DATASET_ROOT),
    'versions': VERSIONS,
    'cameras': CAMERAS,
    'target_source': TARGET_SOURCE,
    'target_mode': TARGET_MODE,
    'target_unit': TARGET_UNIT,
    'target_mean': target_stats['mean'].tolist(),
    'target_std': target_stats['std'].tolist(),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'backbone': BACKBONE,
    'pretrained': PRETRAINED,
    'aggregation': AGGREGATION,
    'num_views': len(CAMERAS),
    'num_port_heads': NUM_PORT_HEADS,
    'main_head': 'pred_rot = softplus(pred_magnitude) * normalize(pred_direction)',
    'direction_loss_weight': DIRECTION_LOSS_WEIGHT,
    'magnitude_loss_weight': MAGNITUDE_LOSS_WEIGHT,
    'direction_eps': DIRECTION_EPS,
    'ema_decay': EMA_DECAY,
    'split': {
        'name': 'episode_holdout',
        'val_ratio': VAL_RATIO,
        'val_count': len(val_samples),
        'val_count_by_version': {version: len(rows) for version, rows in val_samples_by_version.items()},
    },
}

history = fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=EPOCHS,
    weight_dir=WEIGHT_ROOT,
    run_name=RUN_NAME,
    scheduler=scheduler,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    direction_loss_weight=DIRECTION_LOSS_WEIGHT,
    magnitude_loss_weight=MAGNITUDE_LOSS_WEIGHT,
    direction_eps=DIRECTION_EPS,
    ema_decay=EMA_DECAY,
    config=config,
    extra_val_loaders=extra_val_loaders,
)
print(f'best checkpoint: {WEIGHT_ROOT / RUN_NAME / "best.pt"}')
history[-1]


epochs:   2%|▏         | 1/50 [01:22<1:07:08, 82.22s/it, train_loss=0.5085, val_angular=0.1715, val_loss=0.6696]

epoch=001 train_loss=0.5085 val_loss=0.6696 val_mae=0.0872 val_angular=0.1715 val_v3_1_angular=0.2480 val_v2_3_angular=0.0662


epochs:   4%|▍         | 2/50 [01:54<42:19, 52.90s/it, train_loss=0.3843, val_angular=0.1700, val_loss=0.6657]  

epoch=002 train_loss=0.3843 val_loss=0.6657 val_mae=0.0864 val_angular=0.1700 val_v3_1_angular=0.2470 val_v2_3_angular=0.0639


epochs:   6%|▌         | 3/50 [02:26<33:55, 43.32s/it, train_loss=0.2694, val_angular=0.1698, val_loss=0.6626]

...

epoch=030 train_loss=0.0109 val_loss=0.5697 val_mae=0.0827 val_angular=0.1663 val_v3_1_angular=0.2226 val_v2_3_angular=0.0887


epochs:  62%|██████▏   | 31/50 [17:05<09:56, 31.41s/it, train_loss=0.0102, val_angular=0.1669, val_loss=0.5721]

epoch=031 train_loss=0.0102 val_loss=0.5721 val_mae=0.0830 val_angular=0.1669 val_v3_1_angular=0.2231 val_v2_3_angular=0.0897


epochs:  62%|██████▏   | 31/50 [17:22<10:39, 33.64s/it, train_loss=0.0102, val_angular=0.1669, val_loss=0.5721]


KeyboardInterrupt: 

In [9]:
def with_degree_metrics(metrics):
    return {
        **metrics,
        'mae_deg': metrics.get('mae', float('nan')) * 180.0 / math.pi,
        'rmse_deg': metrics.get('rmse', float('nan')) * 180.0 / math.pi,
        'angular_deg': metrics.get('angular', float('nan')) * 180.0 / math.pi,
    }


val_metrics_by_set = {
    'overall': with_degree_metrics(evaluate(
        model,
        val_loader,
        device,
        target_mean=target_stats['mean'],
        target_std=target_stats['std'],
    )),
}
for name, loader in extra_val_loaders.items():
    val_metrics_by_set[name] = with_degree_metrics(evaluate(
        model,
        loader,
        device,
        target_mean=target_stats['mean'],
        target_std=target_stats['std'],
    ))
val_metrics_by_set

{'overall': {'loss': 0.49421156816025996,
  'mae': 0.08748465031385422,
  'rmse': 0.121827632188797,
  'angular': 0.17740924656391144,
  'mae_deg': 5.012501235161699,
  'rmse_deg': 6.980209152490204,
  'angular_deg': 10.164801074707928},
 'v3_1': {'loss': 0.6835733164329918,
  'mae': 0.11325772851705551,
  'rmse': 0.14713077247142792,
  'angular': 0.22939981520175934,
  'mae_deg': 6.489189841265749,
  'rmse_deg': 8.429972299112416,
  'angular_deg': 13.143641232141833},
 'v2_3': {'loss': 0.23357285040148187,
  'mae': 0.05201047286391258,
  'rmse': 0.07399245351552963,
  'angular': 0.10584913939237595,
  'mae_deg': 2.979980585581887,
  'rmse_deg': 4.239455302257779,
  'angular_deg': 6.0647089522750885}}

In [11]:
model.eval()
batch = next(iter(val_loader))
with torch.inference_mode():
    pred_normalized = model(batch['image'].to(device), batch['port_id'].to(device)).cpu()

pred_rad = pred_normalized * target_stats['std'] + target_stats['mean']
target_rad = batch['raw_target']
error_rad = pred_rad - target_rad

preview_deg = torch.cat([pred_rad[:8], target_rad[:8], error_rad[:8]], dim=1) * 180.0 / math.pi
print('columns: pred_roll pred_pitch pred_yaw target_roll target_pitch target_yaw error_roll error_pitch error_yaw')
display(preview_deg)


TypeError: MultimodalOrientationRegressor.forward() missing 1 required positional argument: 'joint_positions'

In [ ]:
correct_count = 0
total_count = 0
before_errors = []
after_errors = []

model.eval()
with torch.inference_mode():
    for eval_batch in val_loader:
        pred_normalized = model(
            eval_batch['image'].to(device),
            eval_batch['port_id'].to(device),
        ).cpu()
        pred_rad = pred_normalized * target_stats['std'] + target_stats['mean']
        target_rad = eval_batch['raw_target']

        before_error = torch.linalg.vector_norm(target_rad, dim=1)
        after_error = torch.linalg.vector_norm(target_rad - pred_rad, dim=1)
        improved = after_error < before_error

        correct_count += int(improved.sum())
        total_count += int(improved.numel())
        before_errors.append(before_error)
        after_errors.append(after_error)

before_error_deg = torch.cat(before_errors) * 180.0 / math.pi
after_error_deg = torch.cat(after_errors) * 180.0 / math.pi
improvement_metrics = {
    'criterion': 'norm(target_delta - pred_delta) < norm(target_delta)',
    'accuracy': correct_count / max(total_count, 1),
    'correct': correct_count,
    'total': total_count,
    'before_error_mean_deg': float(before_error_deg.mean()),
    'after_error_mean_deg': float(after_error_deg.mean()),
    'error_improvement_mean_deg': float((before_error_deg - after_error_deg).mean()),
}
improvement_metrics


{'criterion': 'norm(target_delta - pred_delta) < norm(target_delta)',
 'accuracy': 0.38074398249452956,
 'correct': 174,
 'total': 457,
 'before_error_mean_deg': 9.305950164794922,
 'after_error_mean_deg': 9.754117965698242,
 'error_improvement_mean_deg': -0.4481666088104248}

## v3.1 Three-View Depth Map Feasibility Test

This section tests whether the existing three RGB cameras can produce a usable dense depth map for geometry pretraining. It follows the static camera rig calibration used by `ais_yolo_train/core/triangulation_eval.py`: same image size, FOV-derived intrinsics, camera-link poses, sensor offset, and optical-frame transform.

Note: v3.1 metadata does not store per-sample `CameraInfo` or `base_to_camera` matrices, so this is a feasibility check, not a final calibrated depth generator.


In [ ]:
# Depth feasibility setup: load complete v3.1 three-view samples.
import json
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

DEPTH_VERSION = 'v3.1'
DEPTH_SPLIT = 'train'
DEPTH_ROOT = DATASET_ROOT / DEPTH_VERSION
DEPTH_CAMERAS = ('left', 'center', 'right')

def load_camera_records(version_root: Path, split: str):
    records = []
    for meta_path in sorted((version_root / 'metadata' / split).glob('*.json')):
        record = json.loads(meta_path.read_text())
        record['_metadata_path'] = str(meta_path)
        records.append(record)
    return records

def group_three_view_records(records):
    grouped = {}
    for record in records:
        camera = record.get('camera')
        if camera not in DEPTH_CAMERAS:
            continue
        grouped.setdefault(record['sample_id'], {})[camera] = record
    return {sample_id: views for sample_id, views in grouped.items() if all(camera in views for camera in DEPTH_CAMERAS)}

depth_records = load_camera_records(DEPTH_ROOT, DEPTH_SPLIT)
depth_groups = group_three_view_records(depth_records)
print(f'{DEPTH_VERSION}/{DEPTH_SPLIT}: records={len(depth_records)} complete_3view={len(depth_groups)}')

depth_sample_id = sorted(depth_groups)[0]
depth_views = depth_groups[depth_sample_id]
print('sample_id:', depth_sample_id)
for camera in DEPTH_CAMERAS:
    print(camera, depth_views[camera]['image'])


In [ ]:
# Calibration copied from the triangulation_eval.py convention.
IMAGE_WIDTH = 1152
IMAGE_HEIGHT = 1024
HORIZONTAL_FOV_RAD = 0.8718

CAMERA_LINK_POSES = {
    'center': ((0.0, -0.1077, -0.00719), (0.0, -1.30899630, 1.57079623)),
    'left': ((-0.09326, -0.053843, -0.007188), (0.0, -1.30899630, 0.523599027)),
    'right': ((0.09326, -0.053843, -0.007188), (0.0, -1.30899630, 2.61799343)),
}
SENSOR_IN_CAMERA_LINK = ((0.02174, 0.0, 0.0145), (0.0, 0.0, 0.0))
OPTICAL_IN_SENSOR = ((0.0, 0.0, 0.0), (-1.5708, 0.0, -1.5708))

def rotation_from_rpy(roll: float, pitch: float, yaw: float) -> np.ndarray:
    cr, sr = np.cos(roll), np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cy, sy = np.cos(yaw), np.sin(yaw)
    rx = np.array([[1.0, 0.0, 0.0], [0.0, cr, -sr], [0.0, sr, cr]])
    ry = np.array([[cp, 0.0, sp], [0.0, 1.0, 0.0], [-sp, 0.0, cp]])
    rz = np.array([[cy, -sy, 0.0], [sy, cy, 0.0], [0.0, 0.0, 1.0]])
    return rz @ ry @ rx

def transform_from_xyz_rpy(xyz, rpy) -> np.ndarray:
    transform = np.eye(4, dtype=np.float64)
    transform[:3, :3] = rotation_from_rpy(*rpy)
    transform[:3, 3] = np.asarray(xyz, dtype=np.float64)
    return transform

def camera_to_rig_transforms() -> dict[str, np.ndarray]:
    sensor_to_camera = transform_from_xyz_rpy(*SENSOR_IN_CAMERA_LINK)
    optical_to_sensor = transform_from_xyz_rpy(*OPTICAL_IN_SENSOR)
    transforms = {}
    for name, pose in CAMERA_LINK_POSES.items():
        camera_link_to_rig = transform_from_xyz_rpy(*pose)
        transforms[name] = camera_link_to_rig @ sensor_to_camera @ optical_to_sensor
    return transforms

def intrinsics_from_fov(width=IMAGE_WIDTH, height=IMAGE_HEIGHT, horizontal_fov_rad=HORIZONTAL_FOV_RAD) -> np.ndarray:
    focal = width / (2.0 * np.tan(horizontal_fov_rad / 2.0))
    return np.array([[focal, 0.0, width / 2.0], [0.0, focal, height / 2.0], [0.0, 0.0, 1.0]], dtype=np.float64)

def relative_camera_transform(camera_to_rig: dict[str, np.ndarray], cam1: str, cam2: str):
    # Returns R, T such that X_cam2 = R @ X_cam1 + T.
    cam2_from_cam1 = np.linalg.inv(camera_to_rig[cam2]) @ camera_to_rig[cam1]
    return cam2_from_cam1[:3, :3], cam2_from_cam1[:3, 3]

K_DEPTH = intrinsics_from_fov()
CAMERA_TO_RIG = camera_to_rig_transforms()
print('K=')
print(K_DEPTH)
for a, b in [('center', 'left'), ('center', 'right')]:
    _, t = relative_camera_transform(CAMERA_TO_RIG, a, b)
    print(f'{a}->{b} baseline norm: {np.linalg.norm(t):.4f} m, T={t}')


In [ ]:
# Dense stereo depth from center-left and center-right pairs.
def read_bgr(record):
    path = DEPTH_ROOT / record['image']
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(path)
    return image

def make_rectify_maps(cam1: str, cam2: str, image_size=(IMAGE_WIDTH, IMAGE_HEIGHT)):
    R, T = relative_camera_transform(CAMERA_TO_RIG, cam1, cam2)
    dist = np.zeros(5, dtype=np.float64)
    R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(
        K_DEPTH, dist, K_DEPTH, dist, image_size, R, T, flags=cv2.CALIB_ZERO_DISPARITY, alpha=0.0
    )
    map1x, map1y = cv2.initUndistortRectifyMap(K_DEPTH, dist, R1, P1, image_size, cv2.CV_32FC1)
    map2x, map2y = cv2.initUndistortRectifyMap(K_DEPTH, dist, R2, P2, image_size, cv2.CV_32FC1)
    return (map1x, map1y), (map2x, map2y), Q

def stereo_depth(img1_bgr, img2_bgr, cam1='center', cam2='left', downscale=2):
    if downscale != 1:
        img1_bgr = cv2.resize(img1_bgr, (IMAGE_WIDTH // downscale, IMAGE_HEIGHT // downscale), interpolation=cv2.INTER_AREA)
        img2_bgr = cv2.resize(img2_bgr, (IMAGE_WIDTH // downscale, IMAGE_HEIGHT // downscale), interpolation=cv2.INTER_AREA)
        old_size = (IMAGE_WIDTH, IMAGE_HEIGHT)
        image_size = (old_size[0] // downscale, old_size[1] // downscale)
        k = K_DEPTH.copy()
        k[:2] /= downscale
    else:
        image_size = (IMAGE_WIDTH, IMAGE_HEIGHT)
        k = K_DEPTH

    R, T = relative_camera_transform(CAMERA_TO_RIG, cam1, cam2)
    dist = np.zeros(5, dtype=np.float64)
    R1, R2, P1, P2, Q, *_ = cv2.stereoRectify(k, dist, k, dist, image_size, R, T, flags=cv2.CALIB_ZERO_DISPARITY, alpha=0.0)
    map1x, map1y = cv2.initUndistortRectifyMap(k, dist, R1, P1, image_size, cv2.CV_32FC1)
    map2x, map2y = cv2.initUndistortRectifyMap(k, dist, R2, P2, image_size, cv2.CV_32FC1)
    rect1 = cv2.remap(img1_bgr, map1x, map1y, cv2.INTER_LINEAR)
    rect2 = cv2.remap(img2_bgr, map2x, map2y, cv2.INTER_LINEAR)

    gray1 = cv2.cvtColor(rect1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(rect2, cv2.COLOR_BGR2GRAY)
    num_disp = 16 * 8
    block_size = 5
    stereo = cv2.StereoSGBM_create(
        minDisparity=0,
        numDisparities=num_disp,
        blockSize=block_size,
        P1=8 * block_size * block_size,
        P2=32 * block_size * block_size,
        uniquenessRatio=8,
        speckleWindowSize=80,
        speckleRange=2,
        disp12MaxDiff=1,
        mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY,
    )
    disparity = stereo.compute(gray1, gray2).astype(np.float32) / 16.0
    points_3d = cv2.reprojectImageTo3D(disparity, Q)
    depth = points_3d[:, :, 2]
    valid = np.isfinite(depth) & (disparity > 0.5) & (depth > 0.02) & (depth < 2.0)
    return rect1, rect2, disparity, depth, valid

imgs = {camera: read_bgr(depth_views[camera]) for camera in DEPTH_CAMERAS}
rect_center_l, rect_left, disp_l, depth_l, valid_l = stereo_depth(imgs['center'], imgs['left'], 'center', 'left', downscale=2)
rect_center_r, rect_right, disp_r, depth_r, valid_r = stereo_depth(imgs['center'], imgs['right'], 'center', 'right', downscale=2)

for name, depth, valid in [('center-left', depth_l, valid_l), ('center-right', depth_r, valid_r)]:
    coverage = 100.0 * valid.mean()
    if valid.any():
        print(f'{name}: valid={coverage:.1f}% depth p05/p50/p95={np.percentile(depth[valid], [5, 50, 95])}')
    else:
        print(f'{name}: no valid depth')

fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
axes[0, 0].imshow(cv2.cvtColor(rect_center_l, cv2.COLOR_BGR2RGB)); axes[0, 0].set_title('rectified center')
axes[0, 1].imshow(disp_l, cmap='magma'); axes[0, 1].set_title('disparity center-left')
d0 = np.where(valid_l, depth_l, np.nan)
axes[0, 2].imshow(d0, cmap='viridis', vmin=np.nanpercentile(d0, 5), vmax=np.nanpercentile(d0, 95)); axes[0, 2].set_title('depth center-left')
axes[1, 0].imshow(cv2.cvtColor(rect_center_r, cv2.COLOR_BGR2RGB)); axes[1, 0].set_title('rectified center')
axes[1, 1].imshow(disp_r, cmap='magma'); axes[1, 1].set_title('disparity center-right')
d1 = np.where(valid_r, depth_r, np.nan)
axes[1, 2].imshow(d1, cmap='viridis', vmin=np.nanpercentile(d1, 5), vmax=np.nanpercentile(d1, 95)); axes[1, 2].set_title('depth center-right')
for ax in axes.ravel():
    ax.axis('off')
plt.show()


Interpretation notes:

- If valid coverage is low or depth is noisy, dense stereo is probably not a clean pretraining target from these RGB views alone.
- If the port/connector region has coherent depth while background is noisy, crop/mask-based depth pretraining may still be useful.
- For a production depth target, prefer storing exact `CameraInfo` and `base_to_camera_optical` per sample during collection, matching the `DataCollect2._triangulate_yolo_port` path.
